# 1D CNN Approach

## Imports

fatal: destination path 'MLMAMidtermProject' already exists and is not an empty directory.


In [2]:
import os
import glob
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, precision_recall_curve, auc
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, GlobalAveragePooling1D, Dense, Dropout, Input, BatchNormalization

## Configuring Hyperparameters

In [3]:
FS = 32
WINDOW_SEC = 60
STEP_SEC = 30

WINDOW_SIZE = FS * WINDOW_SEC
STEP_SIZE = FS * STEP_SEC

FEATURES = ['acc_mag', 'EDA', 'HR', 'TEMP', 'EDA_slope', 'HR_slope', 'acc_burst']
TARGET = 'label'

## Loading Data and z-Score Normalization

In [4]:
def load_and_normalize_data(folder_path):
    """
    Loads all nurse CSVs, applies subject-wise Z-score normalization,
    and stores them in a dictionary.
    """
    exclude_nurses = ['CE', 'EG']
    csv_files = glob.glob(os.path.join(folder_path, "processed_nurse_*.csv"))
    csv_files = [f for f in csv_files if not any(f"processed_nurse_{ex}.csv" in os.path.basename(f) for ex in exclude_nurses)]
    subjects_data = {}

    for file in csv_files:
        subject_id = os.path.basename(file).split('.')[0]
        df = pd.read_csv(file)

        # Binarize label: Assuming 0.0 is unstressed, >0 is stressed
        df[TARGET] = (df[TARGET] > 0).astype(int)

        # Calculate the difference between the current moment and 5 seconds ago.
        df['EDA_slope'] = df['EDA'].diff(periods=160).fillna(0)
        df['HR_slope'] = df['HR'].diff(periods=160).fillna(0)

        # Calculate the standard deviation of movement over the last 5 seconds
        df['acc_burst'] = df['acc_mag'].rolling(window=160, min_periods=1).std().fillna(0)

        # Subject-wise Z-Score Normalization
        scaler = StandardScaler()
        df[FEATURES] = scaler.fit_transform(df[FEATURES])

        subjects_data[subject_id] = df

    return subjects_data

## Sliding Window Epochs

In [5]:
def create_windows(df):
    """
    Converts continuous normalized time-series into 3D tensors (Windows, Timesteps, Features).
    """
    data = df[FEATURES].values
    labels = df[TARGET].values

    X, y = [], []
    for start in range(0, len(data) - WINDOW_SIZE, STEP_SIZE):
        end = start + WINDOW_SIZE
        X.append(data[start:end])

        # Label for the window (e.g., mode/majority of the window's labels)
        window_labels = labels[start:end]
        majority_label = 1 if np.mean(window_labels) >= 0.5 else 0
        y.append(majority_label)

    return np.array(X), np.array(y)

## Moderate Undersampling

In [6]:
def moderate_undersample(X, y, target_ratio=0.8):
    """
    Undersamples the majority class (Stressed=1) to reach a specific ratio (e.g., 80:20).
    """
    idx_stressed = np.where(y == 1)[0]
    idx_unstressed = np.where(y == 0)[0]

    # Calculate how many majority samples we need to hit the target_ratio
    # Equation: target_ratio = count_stressed / (count_stressed + count_unstressed)
    num_unstressed = len(idx_unstressed)

    if num_unstressed == 0:
        return X, y # Edge case fallback

    target_stressed_count = int((target_ratio / (1 - target_ratio)) * num_unstressed)

    # Don't oversample if we somehow have fewer stressed than the target requires
    target_stressed_count = min(target_stressed_count, len(idx_stressed))

    # Randomly select majority class indices
    sampled_stressed_idx = np.random.choice(idx_stressed, target_stressed_count, replace=False)

    # Combine and shuffle
    balanced_idx = np.concatenate([sampled_stressed_idx, idx_unstressed])
    np.random.shuffle(balanced_idx)

    return X[balanced_idx], y[balanced_idx]

## Model Creation

In [7]:
def build_1d_cnn(input_shape):
    model = Sequential([
        Input(shape=input_shape),
        Conv1D(filters=32, kernel_size=5, activation='relu'),
        BatchNormalization(),
        MaxPooling1D(pool_size=2),
        Conv1D(filters=64, kernel_size=3, activation='relu'),
        BatchNormalization(),
        GlobalAveragePooling1D(),
        Dense(64, activation='relu', name='dense_1'),
        Dropout(0.4),
        Dense(1, activation='sigmoid', name='output_layer')
    ])

    # Focal Loss Setup (gamma=2.0 focuses on hard examples, alpha handles initial imbalance)
    # focal_loss = tf.keras.losses.BinaryFocalCrossentropy(gamma=2.0, alpha=0.25)

    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

## Cross-Validation (LOSO) and Transfer Learning w/ Metrics

In [8]:
def run_loso_pipeline(subjects_data):
    subject_ids = list(subjects_data.keys())
    all_y_true = []
    all_y_pred_prob = []
    all_y_pred_class = []

    for test_subject in subject_ids:
        tf.keras.backend.clear_session()
        print(f"\n{'='*60}\nLeaving out {test_subject} for Testing/Transfer Learning\n{'='*60}")

        # 1. Prepare Global Training Data (N-1 subjects)
        X_train_global, y_train_global = [], []
        for subj in subject_ids:
            if subj != test_subject:
                X_subj, y_subj = create_windows(subjects_data[subj])
                X_train_global.append(X_subj)
                y_train_global.append(y_subj)

        X_train_global = np.concatenate(X_train_global)
        y_train_global = np.concatenate(y_train_global)

        # Apply moderate undersampling to GLOBAL training set ONLY
        # X_train_bal, y_train_bal = moderate_undersample(X_train_global, y_train_global, target_ratio=0.5)

        # Calculate Class Weights automatically based on the exact imbalance
        weights = compute_class_weight(class_weight='balanced',
                                       classes=np.unique(y_train_global),
                                       y=y_train_global)
        class_weights_dict = {0: weights[0], 1: weights[1]}

        # 2. Train Global Model
        model = build_1d_cnn(input_shape=(WINDOW_SIZE, len(FEATURES)))
        print(f"Training Global Model with Class Weights: {class_weights_dict}...")
        model.fit(X_train_global, y_train_global,
                  epochs=10,
                  batch_size=64,
                  class_weight=class_weights_dict,
                  verbose=0)

        # 3. Transfer Learning Prep (Chronological Split of Left-Out Subject)
        X_test_subj, y_test_subj = create_windows(subjects_data[test_subject])

        # Use first 20% of their shift for fine-tuning, test on remaining 80%
        split_idx = int(len(X_test_subj) * 0.20)
        X_finetune, y_finetune = X_test_subj[:split_idx], y_test_subj[:split_idx]
        X_eval, y_eval = X_test_subj[split_idx:], y_test_subj[split_idx:]

        # Undersample the fine-tuning set as well so it doesn't skew the transfer learning
        # X_finetune_bal, y_finetune_bal = moderate_undersample(X_finetune, y_finetune, target_ratio=0.8)

        # Freeze Convolutional Layers
        for layer in model.layers:
            if isinstance(layer, Conv1D):
                layer.trainable = False

        # Recompile to apply frozen state
        model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
                      loss='binary_crossentropy')

        # 4. Fine-Tune on Target Subject
        print("Checking target subject's early data for fine-tuning...")

        # Ensure we have at least 5 of BOTH classes to prevent a crash and prevent one-sided overfitting
        if len(y_finetune) > 0 and np.sum(y_finetune == 0) >= 5 and np.sum(y_finetune == 1) >= 5:
            # Calculate weights just for this nurse's specific baseline
            ft_weights = compute_class_weight(class_weight='balanced',
                                              classes=np.unique(y_finetune),
                                              y=y_finetune)
            ft_weights_dict = {0: ft_weights[0], 1: ft_weights[1]}

            print("Sufficient minority class found. Fine-tuning...")
            model.fit(X_finetune, y_finetune,
                      epochs=5,
                      batch_size=32,
                      class_weight=ft_weights_dict,
                      verbose=0)
        else:
            print("Not enough 'Unstressed' data in the first 20%. Skipping fine-tuning.")

        # 5. Evaluate on Target Subject's Remaining Data (The true test)
        y_pred_prob = model.predict(X_eval).flatten()

        # Calculate PR curve SPECIFICALLY for the Unstressed (0) class
        precisions, recalls, thresholds = precision_recall_curve((y_eval == 0).astype(int), 1 - y_pred_prob)

        # Calculate F1 scores for the Unstressed class at each threshold
        f1_scores = (2 * precisions * recalls) / (precisions + recalls + 1e-10)

        # Find the threshold that maximizes the F1 score for this specific nurse!
        optimal_idx = np.argmax(f1_scores)

        # Safety catch: precision_recall_curve returns thresholds array that is 1 element shorter than precisions array
        if optimal_idx == len(thresholds):
            optimal_idx -= 1

        # Inverted the probabilities for the function, must invert the threshold back
        optimal_threshold = 1 - thresholds[optimal_idx]

        # Apply their personalized, minority-optimized threshold
        y_pred_class = (y_pred_prob >= optimal_threshold).astype(int)

        # Add to global lists for overall metrics after LOSO loop completes
        all_y_true.extend(y_eval)
        all_y_pred_prob.extend(y_pred_prob)
        all_y_pred_class.extend(y_pred_class)

        # 6. Calculate Specialized Metrics
        cm = confusion_matrix(y_eval, y_pred_class, labels=[0, 1])
        tn, fp, fn, tp = cm.ravel() if cm.size == 4 else (0,0,0,0)

        accuracy = accuracy_score(y_eval, y_pred_class)

        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
        macro_f1 = f1_score(y_eval, y_pred_class, average='macro')

        # Minority F1 (Assuming 0 is the minority Unstressed class)
        minority_f1 = f1_score(y_eval, y_pred_class, pos_label=0, zero_division=0)

        # PR-AUC for the Minority Class
        # Note: precision_recall_curve expects probabilities for the positive class.
        # To get PR-AUC for class 0, we invert the true labels and probabilities.
        precision, recall, _ = precision_recall_curve((y_eval == 0).astype(int), 1 - y_pred_prob)
        pr_auc = auc(recall, precision)

        print("\n--- Evaluation Metrics ---")
        print("Confusion Matrix (TN, FP / FN, TP):\n", cm)
        print(f"Accuracy:                          {accuracy:.4f}")
        print(f"Specificity (Unstressed Accuracy): {specificity:.4f}")
        print(f"Minority F1-Score (Unstressed):    {minority_f1:.4f}")
        print(f"Macro F1-Score:                    {macro_f1:.4f}")
        print(f"Minority PR-AUC:                   {pr_auc:.4f}")

    # Overall Report
    print("\n" + "="*60)
    print("🏆 OVERALL LOSO CROSS-VALIDATION REPORT 🏆")
    print("="*60)

    # Convert lists to numpy arrays
    all_y_true = np.array(all_y_true)
    all_y_pred_prob = np.array(all_y_pred_prob)
    all_y_pred_class = np.array(all_y_pred_class)

    # Overall Confusion Matrix
    cm_overall = confusion_matrix(all_y_true, all_y_pred_class, labels=[0, 1])
    tn, fp, fn, tp = cm_overall.ravel() if cm_overall.size == 4 else (0,0,0,0)

    # Overall Metrics
    overall_accuracy = accuracy_score(all_y_true, all_y_pred_class)
    overall_specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    overall_minority_f1 = f1_score(all_y_true, all_y_pred_class, pos_label=0, zero_division=0)
    overall_macro_f1 = f1_score(all_y_true, all_y_pred_class, average='macro')

    # Overall PR-AUC for Minority Class (0)
    precisions_all, recalls_all, _ = precision_recall_curve((all_y_true == 0).astype(int), 1 - all_y_pred_prob)
    overall_pr_auc = auc(recalls_all, precisions_all)

    print("Overall Confusion Matrix (TN, FP / FN, TP):")
    print(cm_overall)
    print(f"\nTotal Windows Evaluated:     {len(all_y_true)}")
    print(f"Overall Accuracy:            {overall_accuracy:.4f}")
    print(f"Overall Specificity:         {overall_specificity:.4f}")
    print(f"Overall Minority F1 (Calm):  {overall_minority_f1:.4f}")
    print(f"Overall Macro F1:            {overall_macro_f1:.4f}")
    print(f"Overall Minority PR-AUC:     {overall_pr_auc:.4f}")
    print("="*60 + "\n")

## Run Model

In [9]:
subjects_data = load_and_normalize_data("/content/MLMAMidtermProject/data/Aditya")
print(f"Successfully loaded {len(subjects_data)} nurses!")
run_loso_pipeline(subjects_data)

Successfully loaded 13 nurses!

Leaving out processed_nurse_83 for Testing/Transfer Learning
Training Global Model with Class Weights: {0: np.float64(2.191975012013455), 1: np.float64(0.6477563192274922)}...
Checking target subject's early data for fine-tuning...
Sufficient minority class found. Fine-tuning...
36/36 ━━━━━━━━━━━━━━━━━━━━ 2s 32ms/step

--- Evaluation Metrics ---
Confusion Matrix (TN, FP / FN, TP):
 [[ 62  32]
 [446 604]]
Accuracy:                          0.5822
Specificity (Unstressed Accuracy): 0.6596
Minority F1-Score (Unstressed):    0.2060
Macro F1-Score:                    0.4612
Minority PR-AUC:                   0.0989

Leaving out processed_nurse_DF for Testing/Transfer Learning
Training Global Model with Class Weights: {0: np.float64(2.2004122766834633), 1: np.float64(0.6470231681034483)}...
Checking target subject's early data for fine-tuning...
Sufficient minority class found. Fine-tuning...
24/24 ━━━━━━━━━━━━━━━━━━━━ 2s 50ms/step 

--- Evaluation Metrics ---

23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step 

--- Evaluation Metrics ---
Confusion Matrix (TN, FP / FN, TP):
 [[271   1]
 [444   5]]
Accuracy:                          0.3828
Specificity (Unstressed Accuracy): 0.9963
Minority F1-Score (Unstressed):    0.5491
Macro F1-Score:                    0.2856
Minority PR-AUC:                   0.3879

🏆 OVERALL LOSO CROSS-VALIDATION REPORT 🏆
Overall Confusion Matrix (TN, FP / FN, TP):
[[1744   71]
 [4970 1662]]

Total Windows Evaluated:     8447
Overall Accuracy:            0.4032
Overall Specificity:         0.9609
Overall Minority F1 (Calm):  0.4090
Overall Macro F1:            0.4032
Overall Minority PR-AUC:     0.2009

